In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # E6.2 · Catálogo de Reglas Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Mantener de forma gobernada las reglas heurísticas utilizadas
# MAGIC para identificar patrones potencialmente sospechosos en Reviews.
# MAGIC
# MAGIC Este catálogo:
# MAGIC
# MAGIC - NO contiene reglas de Data Quality.
# MAGIC - NO confirma fraude.
# MAGIC - Mantiene definición, propietario, versión, parámetros y pesos.
# MAGIC - Permite activar/desactivar reglas sin perder trazabilidad.
# MAGIC
# MAGIC ## Tablas
# MAGIC
# MAGIC - `workspace.yelp_gov.evaluation_rule`
# MAGIC - `workspace.yelp_gov.evaluation_rule_version`
# MAGIC
# MAGIC ## Estado actual
# MAGIC
# MAGIC - R001-R006: ACTIVE
# MAGIC - R007: DRAFT / experimental


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"
GOV_SCHEMA = "yelp_gov"

GOV = f"{CATALOG}.{GOV_SCHEMA}"

RULE_TABLE = (
    f"{GOV}.evaluation_rule"
)

RULE_VERSION_TABLE = (
    f"{GOV}.evaluation_rule_version"
)

PIPELINE_NAME = (
    "E6_02_catalogo_reglas_ts"
)


print("=" * 72)
print("YELPCONNECT - CATÁLOGO TRUST & SAFETY")
print("=" * 72)

print(
    f"Rule table         : {RULE_TABLE}"
)

print(
    f"Rule version table : {RULE_VERSION_TABLE}"
)

print("=" * 72)


# COMMAND ----------

# ============================================================
# 3. CREAR SCHEMA DE GOBIERNO
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOV_SCHEMA}
    """
)


print(
    f"✔ Schema disponible: {GOV}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 4. Tabla maestra de reglas
# MAGIC
# MAGIC Grano:
# MAGIC
# MAGIC **1 fila = 1 regla Trust & Safety**


# COMMAND ----------

# ============================================================
# 4. EVALUATION RULE
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {RULE_TABLE}
    (
        rule_id              STRING,
        rule_name            STRING,
        rule_type            STRING,
        business_definition  STRING,
        owner                STRING,
        active_flag          BOOLEAN,
        created_at           TIMESTAMP
    )
    USING DELTA
    """
)


print(
    "✔ evaluation_rule disponible."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. Definición de reglas


# COMMAND ----------

# ============================================================
# 5. CATÁLOGO DE REGLAS
# ============================================================

rules = [

    # --------------------------------------------------------
    # R001
    # --------------------------------------------------------

    (
        "R001",
        "Burst Activity",
        "behavioral",
        "Detecta una concentración elevada de Reviews "
        "del mismo usuario dentro de una ventana temporal corta.",
        "Trust & Safety",
        True
    ),


    # --------------------------------------------------------
    # R002
    # --------------------------------------------------------

    (
        "R002",
        "Rating Deviation",
        "behavioral_content",
        "Detecta Reviews cuyo rating presenta una desviación "
        "elevada respecto al rating agregado del Business.",
        "Trust & Safety",
        True
    ),


    # --------------------------------------------------------
    # R003
    # --------------------------------------------------------

    (
        "R003",
        "Repeated Text",
        "content",
        "Detecta reutilización exacta de texto normalizado "
        "en múltiples Reviews.",
        "Trust & Safety",
        True
    ),


    # --------------------------------------------------------
    # R004
    # --------------------------------------------------------

    (
        "R004",
        "New Account High Activity",
        "behavioral",
        "Detecta actividad elevada durante los primeros días "
        "de vida de una cuenta.",
        "Trust & Safety",
        True
    ),


    # --------------------------------------------------------
    # R005
    # --------------------------------------------------------

    (
        "R005",
        "User-Business Concentration",
        "behavioral",
        "Detecta una concentración elevada de Reviews "
        "de un usuario hacia un mismo Business.",
        "Trust & Safety",
        True
    ),


    # --------------------------------------------------------
    # R006
    # --------------------------------------------------------

    (
        "R006",
        "Business Review Spike",
        "temporal",
        "Detecta incrementos anómalos del volumen diario "
        "de Reviews asociado a un Business.",
        "Trust & Safety",
        True
    ),


    # --------------------------------------------------------
    # R007 - EXPERIMENTAL
    # --------------------------------------------------------

    (
        "R007",
        "Cross-Business Repeated Review Pattern",
        "behavioral_content",
        "Detecta usuarios que reutilizan el mismo texto "
        "normalizado en Reviews dirigidas a múltiples "
        "Businesses. La regla se mantiene experimental "
        "hasta validar su precisión y volumen de activación.",
        "Trust & Safety",
        False
    )
]


rule_schema = """
rule_id STRING,
rule_name STRING,
rule_type STRING,
business_definition STRING,
owner STRING,
active_flag BOOLEAN
"""


df_rules = (

    spark

    .createDataFrame(
        rules,
        rule_schema
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)


display(
    df_rules.orderBy(
        "rule_id"
    )
)


# COMMAND ----------

# ============================================================
# 6. MERGE EVALUATION RULE
#
# created_at se conserva para reglas ya existentes.
# ============================================================

df_rules.createOrReplaceTempView(
    "src_evaluation_rule"
)


spark.sql(
    f"""
    MERGE INTO
        {RULE_TABLE} AS target

    USING
        src_evaluation_rule AS source

    ON
        target.rule_id =
        source.rule_id

    WHEN MATCHED THEN

        UPDATE SET

            target.rule_name =
                source.rule_name,

            target.rule_type =
                source.rule_type,

            target.business_definition =
                source.business_definition,

            target.owner =
                source.owner,

            target.active_flag =
                source.active_flag

    WHEN NOT MATCHED THEN

        INSERT
        (
            rule_id,
            rule_name,
            rule_type,
            business_definition,
            owner,
            active_flag,
            created_at
        )

        VALUES
        (
            source.rule_id,
            source.rule_name,
            source.rule_type,
            source.business_definition,
            source.owner,
            source.active_flag,
            source.created_at
        )
    """
)


print(
    "✔ Catálogo maestro actualizado."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Versionado de reglas
# MAGIC
# MAGIC Grano:
# MAGIC
# MAGIC **1 fila = 1 versión de una regla**


# COMMAND ----------

# ============================================================
# 7. EVALUATION RULE VERSION
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {RULE_VERSION_TABLE}
    (
        rule_version_id  STRING,
        rule_id          STRING,
        version          STRING,
        description      STRING,
        parameters_json  STRING,
        threshold        DOUBLE,
        weight           DOUBLE,
        valid_from       TIMESTAMP,
        status           STRING
    )
    USING DELTA
    """
)


print(
    "✔ evaluation_rule_version disponible."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. Versiones vigentes
# MAGIC
# MAGIC R007 se registra como:
# MAGIC
# MAGIC - versión `0.1`
# MAGIC - `DRAFT`
# MAGIC - peso `0.0`
# MAGIC
# MAGIC Esto permite medirla sin alterar todavía el Risk Score.


# COMMAND ----------

# ============================================================
# 8. CATÁLOGO DE VERSIONES
# ============================================================

rule_versions = [

    # --------------------------------------------------------
    # R001
    # --------------------------------------------------------

    (
        "R001-v1.0",
        "R001",
        "1.0",
        "8 o más Reviews del mismo usuario "
        "en una ventana trailing de 30 minutos.",
        """
        {
            "window_minutes": 30,
            "min_reviews": 8
        }
        """,
        8.0,
        0.20,
        "ACTIVE"
    ),


    # --------------------------------------------------------
    # R002
    # --------------------------------------------------------

    (
        "R002-v1.0",
        "R002",
        "1.0",
        "Desviación absoluta de al menos 3 estrellas "
        "respecto al rating agregado del Business.",
        """
        {
            "rating_difference": 3.0
        }
        """,
        3.0,
        0.15,
        "ACTIVE"
    ),


    # --------------------------------------------------------
    # R003
    # --------------------------------------------------------

    (
        "R003-v1.0",
        "R003",
        "1.0",
        "Texto normalizado de al menos 40 caracteres "
        "repetido al menos 3 veces.",
        """
        {
            "min_repetitions": 3,
            "min_text_length": 40
        }
        """,
        3.0,
        0.20,
        "ACTIVE"
    ),


    # --------------------------------------------------------
    # R004
    # --------------------------------------------------------

    (
        "R004-v1.0",
        "R004",
        "1.0",
        "Cuenta con antigüedad máxima de 30 días "
        "y al menos 5 Reviews dentro del dataset.",
        """
        {
            "account_age_days": 30,
            "min_reviews": 5
        }
        """,
        5.0,
        0.15,
        "ACTIVE"
    ),


    # --------------------------------------------------------
    # R005
    # --------------------------------------------------------

    (
        "R005-v1.0",
        "R005",
        "1.0",
        "Al menos 80 por ciento de las Reviews del usuario "
        "concentradas en un mismo Business, con mínimo "
        "3 Reviews hacia dicho Business.",
        """
        {
            "concentration": 0.80,
            "min_business_reviews": 3
        }
        """,
        0.80,
        0.15,
        "ACTIVE"
    ),


    # --------------------------------------------------------
    # R006
    # --------------------------------------------------------

    (
        "R006-v1.0",
        "R006",
        "1.0",
        "Al menos 10 Reviews en un día y volumen "
        "igual o superior a 3 veces el promedio diario "
        "activo del Business.",
        """
        {
            "min_daily_reviews": 10,
            "spike_ratio": 3.0
        }
        """,
        3.0,
        0.15,
        "ACTIVE"
    ),


    # --------------------------------------------------------
    # R007 - EXPERIMENTAL
    #
    # No participa todavía en el Risk Score.
    # --------------------------------------------------------

    (
        "R007-v0.1",
        "R007",
        "0.1",
        "Usuario reutiliza el mismo texto normalizado "
        "en Reviews dirigidas a por lo menos "
        "2 Businesses diferentes.",
        """
        {
            "min_distinct_businesses": 2,
            "min_text_repetitions": 2,
            "experimental": true
        }
        """,
        2.0,
        0.0,
        "DRAFT"
    )
]


version_schema = """
rule_version_id STRING,
rule_id STRING,
version STRING,
description STRING,
parameters_json STRING,
threshold DOUBLE,
weight DOUBLE,
status STRING
"""


df_rule_versions = (

    spark

    .createDataFrame(
        rule_versions,
        version_schema
    )

    .withColumn(
        "parameters_json",

        F.regexp_replace(
            F.trim(
                F.col(
                    "parameters_json"
                )
            ),
            r"\s+",
            " "
        )
    )

    .withColumn(
        "valid_from",
        F.current_timestamp()
    )
)


display(
    df_rule_versions

    .select(
        "rule_version_id",
        "rule_id",
        "version",
        "threshold",
        "weight",
        "status",
        "parameters_json"
    )

    .orderBy(
        "rule_id"
    )
)


# COMMAND ----------

# ============================================================
# 9. MERGE VERSIONES
#
# valid_from no se modifica para versiones ya existentes.
# ============================================================

df_rule_versions.createOrReplaceTempView(
    "src_evaluation_rule_version"
)


spark.sql(
    f"""
    MERGE INTO
        {RULE_VERSION_TABLE} AS target

    USING
        src_evaluation_rule_version AS source

    ON
        target.rule_version_id =
        source.rule_version_id

    WHEN MATCHED THEN

        UPDATE SET

            target.rule_id =
                source.rule_id,

            target.version =
                source.version,

            target.description =
                source.description,

            target.parameters_json =
                source.parameters_json,

            target.threshold =
                source.threshold,

            target.weight =
                source.weight,

            target.status =
                source.status

    WHEN NOT MATCHED THEN

        INSERT
        (
            rule_version_id,
            rule_id,
            version,
            description,
            parameters_json,
            threshold,
            weight,
            valid_from,
            status
        )

        VALUES
        (
            source.rule_version_id,
            source.rule_id,
            source.version,
            source.description,
            source.parameters_json,
            source.threshold,
            source.weight,
            source.valid_from,
            source.status
        )
    """
)


print(
    "✔ Versiones actualizadas."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 10. Validación de integridad del catálogo


# COMMAND ----------

# ============================================================
# 10. VERSIONES SIN REGLA PADRE
# ============================================================

orphan_versions = (

    spark.table(
        RULE_VERSION_TABLE
    )
    .alias("v")

    .join(

        spark.table(
            RULE_TABLE
        )
        .alias("r"),

        F.col(
            "v.rule_id"
        )
        ==
        F.col(
            "r.rule_id"
        ),

        "left_anti"
    )

    .count()
)


if orphan_versions > 0:

    raise Exception(
        f"""
        ❌ Se encontraron {orphan_versions}
        versiones sin regla padre.
        """
    )


print(
    "✔ Todas las versiones tienen regla padre."
)


# COMMAND ----------

# ============================================================
# 11. VALIDAR REGLAS ACTIVAS
# ============================================================

active_without_active_version = (

    spark.table(
        RULE_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "r.active_flag"
        )
        ==
        True
    )

    .join(

        spark.table(
            RULE_VERSION_TABLE
        )
        .alias("v")

        .filter(
            F.col(
                "v.status"
            )
            ==
            "ACTIVE"
        ),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "left"
    )

    .filter(
        F.col(
            "v.rule_version_id"
        ).isNull()
    )

    .count()
)


if active_without_active_version > 0:

    raise Exception(
        f"""
        ❌ Existen {active_without_active_version}
        reglas activas sin versión ACTIVE.
        """
    )


print(
    "✔ Reglas activas con versión vigente."
)


# COMMAND ----------

# ============================================================
# 12. VALIDAR PESOS ACTIVOS
# ============================================================

active_weight = (

    spark.table(
        RULE_VERSION_TABLE
    )

    .filter(
        F.col(
            "status"
        )
        ==
        "ACTIVE"
    )

    .agg(

        F.round(
            F.sum(
                "weight"
            ),
            4
        )
        .alias(
            "total_weight"
        )
    )

    .first()
    .total_weight
)


print(
    f"Peso total reglas ACTIVE: "
    f"{active_weight}"
)


if active_weight is None:

    raise Exception(
        """
        ❌ No existen versiones ACTIVE.
        """
    )


# El modelo actual R001-R006 suma exactamente 1.0.

if abs(
    active_weight - 1.0
) > 0.0001:

    print(
        "⚠ ADVERTENCIA: "
        "los pesos de reglas ACTIVE "
        "no suman 1.0."
    )

else:

    print(
        "✔ Pesos ACTIVE suman 1.0."
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 13. Estado completo del catálogo


# COMMAND ----------

catalog_view = (

    spark.table(
        RULE_TABLE
    )
    .alias("r")

    .join(

        spark.table(
            RULE_VERSION_TABLE
        )
        .alias("v"),

        F.col(
            "r.rule_id"
        )
        ==
        F.col(
            "v.rule_id"
        ),

        "left"
    )

    .select(

        F.col(
            "r.rule_id"
        ),

        F.col(
            "r.rule_name"
        ),

        F.col(
            "r.rule_type"
        ),

        F.col(
            "r.owner"
        ),

        F.col(
            "r.active_flag"
        ),

        F.col(
            "v.rule_version_id"
        ),

        F.col(
            "v.version"
        ),

        F.col(
            "v.threshold"
        ),

        F.col(
            "v.weight"
        ),

        F.col(
            "v.status"
        ),

        F.col(
            "v.valid_from"
        )
    )

    .orderBy(
        "rule_id",
        "version"
    )
)


display(
    catalog_view
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 14. Reglas actualmente ejecutables


# COMMAND ----------

display(

    catalog_view

    .filter(
        (
            F.col(
                "active_flag"
            )
            ==
            True
        )

        &

        (
            F.col(
                "status"
            )
            ==
            "ACTIVE"
        )
    )

    .select(
        "rule_id",
        "rule_name",
        "rule_version_id",
        "threshold",
        "weight"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 15. Reglas experimentales


# COMMAND ----------

display(

    catalog_view

    .filter(
        F.col(
            "status"
        )
        ==
        "DRAFT"
    )

    .select(
        "rule_id",
        "rule_name",
        "rule_version_id",
        "threshold",
        "weight",
        "status",
        "active_flag"
    )
)


# COMMAND ----------

# ============================================================
# 16. RESUMEN
# ============================================================

rule_count = (
    spark.table(
        RULE_TABLE
    )
    .count()
)


active_rule_count = (

    spark.table(
        RULE_TABLE
    )

    .filter(
        F.col(
            "active_flag"
        )
        ==
        True
    )

    .count()
)


draft_version_count = (

    spark.table(
        RULE_VERSION_TABLE
    )

    .filter(
        F.col(
            "status"
        )
        ==
        "DRAFT"
    )

    .count()
)


print()
print("=" * 72)
print("CATÁLOGO TRUST & SAFETY ACTUALIZADO")
print("=" * 72)

print(
    f"Reglas registradas    : {rule_count}"
)

print(
    f"Reglas activas        : {active_rule_count}"
)

print(
    f"Versiones DRAFT       : {draft_version_count}"
)

print(
    f"Peso reglas ACTIVE    : {active_weight}"
)

print()
print(
    "R007 permanece experimental "
    "y NO participa todavía en el Risk Score."
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done
# MAGIC
# MAGIC - [ ] R001-R006 registradas como activas.
# MAGIC - [ ] R001-R006 tienen versión ACTIVE.
# MAGIC - [ ] R007 registrada.
# MAGIC - [ ] R007-v0.1 permanece DRAFT.
# MAGIC - [ ] R007 tiene peso 0.0.
# MAGIC - [ ] Todas las versiones tienen regla padre.
# MAGIC - [ ] Las reglas activas tienen versión activa.
# MAGIC - [ ] Los pesos activos suman 1.0.